# Visual Search System - Data Engineering & Feature Engineering

This notebook covers the data engineering and feature engineering aspects of building a visual search system. We'll explore available data sources, image preprocessing operations, and considerations for working with image embeddings.

**Learning Objectives:**
- Understand the data sources available for training a visual search model
- Learn essential image preprocessing operations
- Explore feature engineering considerations for image embeddings

In [ ]:
# Standard imports for this notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from typing import List, Dict, Optional, Tuple

---

## 1. Available Data Sources

A visual search system relies on multiple data sources to function effectively. Let's explore the key data tables that would be available in a Pinterest-like platform.

### Overview of Data Sources

| Table | Description | Primary Use |
|-------|-------------|-------------|
| Images | Core image repository with metadata | Searchable content |
| Users | User profiles and demographics | Context and analytics |
| User-Image Interactions | Clicks, views, saves | Training signal |

### 1.1 Images Table

The Images table is the core data source containing all images in the platform.

**Schema:**

| Column | Type | Description |
|--------|------|-------------|
| `image_id` | STRING | Unique identifier for each image |
| `owner_id` | STRING | User ID who uploaded the image |
| `upload_time` | TIMESTAMP | When the image was uploaded |
| `manual_tags` | ARRAY[STRING] | User-provided tags/labels |
| `image_url` | STRING | Storage location of the image |
| `width` | INT | Image width in pixels |
| `height` | INT | Image height in pixels |
| `file_size_bytes` | INT | Size of image file |

In [ ]:
# Sample Images Table
np.random.seed(42)

n_images = 10
sample_images = {
    'image_id': [f'img_{i:06d}' for i in range(n_images)],
    'owner_id': [f'user_{np.random.randint(1, 100):04d}' for _ in range(n_images)],
    'upload_time': pd.date_range('2024-01-01', periods=n_images, freq='D'),
    'manual_tags': [
        ['sunset', 'beach', 'travel'],
        ['dog', 'golden retriever', 'pet'],
        ['food', 'pasta', 'italian'],
        ['architecture', 'modern', 'building'],
        ['fashion', 'dress', 'summer'],
        ['nature', 'forest', 'hiking'],
        ['car', 'vintage', 'classic'],
        ['art', 'painting', 'abstract'],
        ['coffee', 'cafe', 'morning'],
        ['flower', 'garden', 'spring']
    ],
    'width': np.random.randint(800, 4000, n_images),
    'height': np.random.randint(600, 3000, n_images),
    'file_size_bytes': np.random.randint(100000, 5000000, n_images)
}

df_images = pd.DataFrame(sample_images)
print("Sample Images Table")
print("=" * 80)
print(df_images.to_string(index=False))

In [ ]:
# Image Table Statistics
print("\nImage Table Statistics (Simulated at Scale)")
print("=" * 50)

scale_stats = {
    'Total Images': '150 billion',
    'Daily New Uploads': '10 million',
    'Avg Image Size': '1.2 MB',
    'Total Storage': '~180 PB (raw images)',
    'Most Common Tags': ['fashion', 'food', 'travel', 'home', 'art'],
    'Avg Tags per Image': 3.2
}

for key, value in scale_stats.items():
    print(f"  {key}: {value}")

### 1.2 Users Table

The Users table contains profile information about platform users.

**Schema:**

| Column | Type | Description |
|--------|------|-------------|
| `user_id` | STRING | Unique identifier for each user |
| `username` | STRING | Display name |
| `age` | INT | User's age |
| `gender` | STRING | User's gender |
| `city` | STRING | City of residence |
| `country` | STRING | Country of residence |
| `email` | STRING | Email address (for account purposes) |
| `signup_date` | TIMESTAMP | When user created account |

In [ ]:
# Sample Users Table
n_users = 8
cities = ['New York', 'San Francisco', 'Los Angeles', 'Chicago', 'Seattle', 'Austin', 'Boston', 'Denver']
countries = ['USA', 'USA', 'USA', 'USA', 'USA', 'USA', 'USA', 'USA']
genders = ['M', 'F', 'M', 'F', 'F', 'M', 'F', 'M']

sample_users = {
    'user_id': [f'user_{i:04d}' for i in range(1, n_users + 1)],
    'username': ['alex_photos', 'bella_art', 'charlie_design', 'diana_travel', 
                 'emma_style', 'frank_tech', 'grace_food', 'henry_nature'],
    'age': [28, 34, 25, 42, 31, 29, 38, 45],
    'gender': genders,
    'city': cities,
    'country': countries,
    'email': [f'{name.split("_")[0]}@email.com' for name in 
             ['alex_photos', 'bella_art', 'charlie_design', 'diana_travel',
              'emma_style', 'frank_tech', 'grace_food', 'henry_nature']],
    'signup_date': pd.date_range('2020-01-01', periods=n_users, freq='45D')
}

df_users = pd.DataFrame(sample_users)
print("Sample Users Table")
print("=" * 100)
print(df_users.to_string(index=False))

### 1.3 User-Image Interactions Table

The interactions table is **critical for training** the visual search model. It captures user behavior when interacting with search results.

**Schema:**

| Column | Type | Description |
|--------|------|-------------|
| `user_id` | STRING | User who performed the search |
| `query_image_id` | STRING | The image used as the search query |
| `displayed_image_id` | STRING | An image shown in search results |
| `position` | INT | Position in the result list (1-indexed) |
| `interaction_type` | STRING | Type of interaction (click, impression, save) |
| `location` | STRING | Geographic location of user |
| `timestamp` | TIMESTAMP | When the interaction occurred |

In [ ]:
# Sample User-Image Interactions Table
np.random.seed(123)

n_interactions = 15
interaction_types = ['impression', 'impression', 'click', 'impression', 'save']

sample_interactions = {
    'user_id': [f'user_{np.random.randint(1, 100):04d}' for _ in range(n_interactions)],
    'query_image_id': np.random.choice(['img_000001', 'img_000002', 'img_000003'], n_interactions),
    'displayed_image_id': [f'img_{np.random.randint(100, 999):06d}' for _ in range(n_interactions)],
    'position': np.random.randint(1, 20, n_interactions),
    'interaction_type': np.random.choice(interaction_types, n_interactions),
    'location': np.random.choice(['US', 'UK', 'DE', 'FR', 'JP', 'BR'], n_interactions),
    'timestamp': pd.date_range('2024-06-01 10:00:00', periods=n_interactions, freq='15min')
}

df_interactions = pd.DataFrame(sample_interactions)
df_interactions = df_interactions.sort_values(['query_image_id', 'position']).reset_index(drop=True)

print("Sample User-Image Interactions Table")
print("=" * 100)
print(df_interactions.to_string(index=False))

In [ ]:
# Analyze interaction patterns
print("\nInteraction Type Distribution")
print("=" * 40)
interaction_counts = df_interactions['interaction_type'].value_counts()
for itype, count in interaction_counts.items():
    pct = count / len(df_interactions) * 100
    print(f"  {itype.capitalize():12s}: {count:3d} ({pct:.1f}%)")

print("\nTraining Signal Interpretation:")
print("  - 'click' or 'save' -> Positive pair (query_img, displayed_img)")
print("  - 'impression' only -> Weak negative pair")
print("  - Position matters: lower position + click = stronger positive signal")

---

## 2. Image Preprocessing Operations

Before feeding images to a neural network, we need to apply several preprocessing steps to ensure consistency and optimal performance.

### Why Preprocessing Matters

1. **Consistency**: Neural networks expect fixed input dimensions
2. **Normalization**: Helps with training stability and convergence
3. **Color Standardization**: Ensures consistent color representation
4. **Efficiency**: Reduces computational overhead

### 2.1 Resizing (e.g., 224x224)

Most pre-trained vision models expect a specific input size, commonly:
- **224x224** (ResNet, VGG, EfficientNet-B0)
- **299x299** (Inception)
- **384x384** (ViT-Base)
- **518x518** (EfficientNet-B5)

**Resizing Strategies:**

| Strategy | Description | Pros | Cons |
|----------|-------------|------|------|
| Stretch | Resize directly to target | Simple | Distorts aspect ratio |
| Crop | Center crop to target | Preserves ratio | May lose important content |
| Pad | Resize + pad to square | Preserves ratio | Adds empty space |
| Letterbox | Resize maintaining ratio + pad | Best of both | Slightly complex |

In [ ]:
# Demonstration: Different resizing strategies
def visualize_resize_strategies():
    """Visualize different image resizing strategies."""
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    
    original_w, original_h = 800, 600
    target_size = 224
    
    strategies = [
        ('Original\n(800x600)', (original_w/100, original_h/100), '#3498db'),
        ('Stretch\n(224x224)', (2.24, 2.24), '#e74c3c'),
        ('Center Crop\n(224x224)', (2.24, 2.24), '#2ecc71'),
        ('Letterbox\n(224x224)', (2.24, 2.24), '#9b59b6')
    ]
    
    for ax, (title, size, color) in zip(axes, strategies):
        rect = plt.Rectangle((0.5 - size[0]/20, 0.5 - size[1]/20), 
                            size[0]/10, size[1]/10,
                            facecolor=color, alpha=0.3, 
                            edgecolor=color, linewidth=2)
        ax.add_patch(rect)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_title(title, fontsize=11, fontweight='bold')
        ax.set_aspect('equal')
        ax.axis('off')
    
    plt.suptitle('Image Resizing Strategies', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_resize_strategies()

print("\nRecommendation: Use letterbox/pad for visual search to preserve aspect ratio")

### 2.2 Scaling Pixel Values (0 to 1)

Raw image pixels are typically in the range [0, 255] (8-bit integers). Neural networks work better with normalized floating-point values.

**Simple Scaling:**

x_scaled = x_raw / 255

This maps pixel values from [0, 255] to [0, 1].

In [ ]:
# Demonstration: Pixel value scaling
print("Pixel Value Scaling")
print("=" * 50)

raw_pixels = np.array([0, 64, 128, 192, 255], dtype=np.uint8)
scaled_pixels = raw_pixels.astype(np.float32) / 255.0

print("\nBefore Scaling (uint8, range 0-255):")
print(f"  {raw_pixels}")

print("\nAfter Scaling (float32, range 0-1):")
print(f"  {scaled_pixels}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(len(raw_pixels)), raw_pixels, color='steelblue')
axes[0].set_title('Raw Pixel Values (0-255)', fontweight='bold')
axes[0].set_ylabel('Value')
axes[0].set_ylim(0, 300)

axes[1].bar(range(len(scaled_pixels)), scaled_pixels, color='coral')
axes[1].set_title('Scaled Pixel Values (0-1)', fontweight='bold')
axes[1].set_ylabel('Value')
axes[1].set_ylim(0, 1.2)

plt.tight_layout()
plt.show()

### 2.3 Z-score Normalization

For pre-trained models, we often apply z-score normalization using ImageNet statistics:

x_normalized = (x - mean) / std

**ImageNet Statistics:**
- Mean: [0.485, 0.456, 0.406] (RGB channels)
- Std: [0.229, 0.224, 0.225] (RGB channels)

This centers the data around zero with unit variance, matching the distribution the model was trained on.

In [ ]:
# Demonstration: Z-score normalization
print("Z-score Normalization with ImageNet Statistics")
print("=" * 55)

imagenet_mean = np.array([0.485, 0.456, 0.406])
imagenet_std = np.array([0.229, 0.224, 0.225])

print(f"\nImageNet Mean (RGB): {imagenet_mean}")
print(f"ImageNet Std (RGB):  {imagenet_std}")

sample_pixel = np.array([0.6, 0.4, 0.3])
normalized_pixel = (sample_pixel - imagenet_mean) / imagenet_std

print(f"\nSample pixel (scaled): {sample_pixel}")
print(f"After z-score norm:    {normalized_pixel}")

print("\nFormula for each channel:")
for i, channel in enumerate(['Red', 'Green', 'Blue']):
    print(f"  {channel}: ({sample_pixel[i]:.3f} - {imagenet_mean[i]:.3f}) / {imagenet_std[i]:.3f} = {normalized_pixel[i]:.3f}")

### 2.4 Consistent Color Mode (RGB/CMYK)

Images can come in different color modes:
- **RGB** (Red, Green, Blue) - Standard for digital displays
- **RGBA** (RGB + Alpha) - Includes transparency
- **CMYK** (Cyan, Magenta, Yellow, Key/Black) - For printing
- **Grayscale** (L) - Single channel
- **P** (Palette) - Indexed colors

**For visual search, we standardize to RGB.**

In [ ]:
# Color mode handling
print("Color Mode Standardization")
print("=" * 50)

color_modes = {
    'RGB': {'channels': 3, 'description': 'Standard color (Red, Green, Blue)', 'action': 'Keep as-is'},
    'RGBA': {'channels': 4, 'description': 'Color with transparency', 'action': 'Convert to RGB (drop alpha)'},
    'CMYK': {'channels': 4, 'description': 'Print color space', 'action': 'Convert to RGB'},
    'L': {'channels': 1, 'description': 'Grayscale', 'action': 'Convert to RGB (repeat channel)'},
    'P': {'channels': 1, 'description': 'Palette/indexed colors', 'action': 'Convert to RGB'},
}

print("\nColor Mode Handling:")
print("-" * 80)
for mode, info in color_modes.items():
    print(f"  {mode:5s} | {info['channels']} ch | {info['description']:30s} | {info['action']}")

---

## 3. Feature Engineering Considerations

In visual search, "feature engineering" takes on a different meaning compared to traditional ML. Instead of hand-crafting features, we focus on:

1. **Working with image embeddings** (learned representations)
2. **Handling interaction data** for training
3. **Data augmentation** strategies

### 3.1 Working with Image Embeddings

**What are Embeddings?**

Embeddings are dense vector representations of images learned by neural networks. These vectors capture semantic information about the image content.

**Embedding Characteristics:**

| Property | Typical Value | Notes |
|----------|---------------|-------|
| Dimensionality | 256-2048 | Higher = more capacity, but more storage |
| Data Type | float32 | Can be quantized to int8 for efficiency |
| Normalization | L2-normalized | Makes cosine similarity = dot product |
| Storage per Image | 1-8 KB | At scale, this adds up quickly |

In [ ]:
# Embedding dimension analysis
print("Embedding Dimension Trade-offs")
print("=" * 60)

embedding_configs = [
    {'dim': 128, 'model': 'MobileNet-v3-small'},
    {'dim': 256, 'model': 'EfficientNet-B0'},
    {'dim': 512, 'model': 'ResNet-50'},
    {'dim': 768, 'model': 'ViT-Base'},
    {'dim': 1024, 'model': 'ResNet-152'},
    {'dim': 2048, 'model': 'ResNet-50 (before pool)'},
]

total_images = 150_000_000_000

print(f"\nStorage Analysis (for {total_images:,} images):")
print("-" * 60)

for config in embedding_configs:
    dim = config['dim']
    model = config['model']
    
    bytes_per_embedding = dim * 4
    total_bytes = total_images * bytes_per_embedding
    total_tb = total_bytes / (1024 ** 4)
    
    print(f"  {dim:4d}-d ({model:25s}): {bytes_per_embedding:,} bytes/img -> {total_tb:,.0f} TB total")

In [ ]:
# Demonstrate embedding operations
print("\nCommon Embedding Operations")
print("=" * 50)

np.random.seed(42)
embedding_dim = 256

query_emb = np.random.randn(embedding_dim).astype(np.float32)
similar_emb = query_emb + np.random.randn(embedding_dim).astype(np.float32) * 0.1
different_emb = np.random.randn(embedding_dim).astype(np.float32)

def l2_normalize(x):
    return x / np.linalg.norm(x)

query_emb_norm = l2_normalize(query_emb)
similar_emb_norm = l2_normalize(similar_emb)
different_emb_norm = l2_normalize(different_emb)

print("\n1. L2 Normalization:")
print(f"   Before: ||query|| = {np.linalg.norm(query_emb):.4f}")
print(f"   After:  ||query|| = {np.linalg.norm(query_emb_norm):.4f}")

print("\n2. Dot Product Similarity (after L2 norm = cosine similarity):")
sim_score = np.dot(query_emb_norm, similar_emb_norm)
diff_score = np.dot(query_emb_norm, different_emb_norm)
print(f"   Query vs Similar: {sim_score:.4f}")
print(f"   Query vs Different: {diff_score:.4f}")

print("\n3. Quantization (float32 -> int8):")
quantized = np.clip(query_emb_norm * 127, -128, 127).astype(np.int8)
print(f"   Original size: {query_emb.nbytes} bytes")
print(f"   Quantized size: {quantized.nbytes} bytes")
print(f"   Compression ratio: {query_emb.nbytes / quantized.nbytes:.1f}x")

### 3.2 Handling Interaction Data

Interaction data from user behavior needs to be processed into training signals:

**Data Processing Steps:**

1. **Filter valid interactions** - Remove bots, invalid sessions
2. **Create training pairs** - (query_image, result_image, label)
3. **Handle position bias** - Account for users clicking top results more often
4. **Aggregate signals** - Multiple interactions -> confidence scores

In [ ]:
# Create training dataset from interactions
print("Creating Training Dataset from Interactions")
print("=" * 55)

np.random.seed(456)
n_sessions = 100

sessions = []
for session_id in range(n_sessions):
    query_id = f'query_{np.random.randint(1000, 9999)}'
    n_results = np.random.randint(5, 20)
    
    for position in range(1, n_results + 1):
        result_id = f'result_{np.random.randint(10000, 99999)}'
        click_prob = 0.3 / (position ** 0.5)
        clicked = np.random.random() < click_prob
        
        sessions.append({
            'session_id': session_id,
            'query_id': query_id,
            'result_id': result_id,
            'position': position,
            'clicked': clicked
        })

df_sessions = pd.DataFrame(sessions)

print(f"\nTotal interactions: {len(df_sessions):,}")
print(f"Unique sessions: {df_sessions['session_id'].nunique()}")
print(f"Click rate: {df_sessions['clicked'].mean():.2%}")

position_ctr = df_sessions.groupby('position')['clicked'].mean()

print("\nClick-through Rate by Position (Position Bias):")
for pos in [1, 2, 3, 5, 10]:
    if pos in position_ctr.index:
        print(f"  Position {pos:2d}: {position_ctr[pos]:.2%}")

In [ ]:
# Visualize position bias
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(position_ctr.index[:15], position_ctr.values[:15], color='steelblue')
axes[0].set_xlabel('Position', fontsize=12)
axes[0].set_ylabel('Click-through Rate', fontsize=12)
axes[0].set_title('Position Bias in Click Data', fontsize=14, fontweight='bold')
axes[0].axhline(y=df_sessions['clicked'].mean(), color='red', linestyle='--', label='Avg CTR')
axes[0].legend()

positive_count = df_sessions['clicked'].sum()
negative_count = len(df_sessions) - positive_count

axes[1].pie([positive_count, negative_count], 
           labels=['Positive (Clicked)', 'Negative (Impression only)'],
           colors=['#2ecc71', '#e74c3c'],
           autopct='%1.1f%%',
           explode=(0.05, 0))
axes[1].set_title('Training Data Label Distribution', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nNote: Heavy class imbalance - need to handle carefully during training")

### 3.3 Data Augmentation Strategies

Data augmentation increases training data diversity and improves model robustness.

**Common Augmentations for Visual Search:**

| Augmentation | Description | Use Case |
|--------------|-------------|----------|
| Horizontal Flip | Mirror image horizontally | General invariance |
| Random Crop | Crop random regions | Partial object matching |
| Color Jitter | Adjust brightness/contrast/saturation | Lighting invariance |
| Rotation | Small angle rotations | Orientation invariance |
| Scale | Random scaling | Size invariance |

In [ ]:
# Visualize augmentation effects
def visualize_augmentations():
    """Visualize different augmentation types."""
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    
    augmentations = [
        ('Original', '#3498db'),
        ('Horizontal\nFlip', '#2ecc71'),
        ('Random\nCrop', '#e74c3c'),
        ('Rotation\n(+/-15 deg)', '#9b59b6'),
        ('Brightness\n(+20%)', '#f39c12'),
        ('Contrast\n(+20%)', '#1abc9c'),
        ('Saturation\n(+20%)', '#e67e22'),
        ('Combined', '#34495e')
    ]
    
    for ax, (name, color) in zip(axes.flat, augmentations):
        rect = plt.Rectangle((0.1, 0.1), 0.8, 0.8, 
                            facecolor=color, alpha=0.5,
                            edgecolor='black', linewidth=2)
        ax.add_patch(rect)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_title(name, fontsize=11, fontweight='bold')
        ax.axis('off')
    
    plt.suptitle('Common Data Augmentations for Visual Search', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_augmentations()

print("\nKey Insight: Augmentations should preserve visual similarity")
print("   (A flipped cat is still similar to the original cat)")

---

## Summary

In this notebook, we covered the data and feature engineering aspects of visual search:

### Key Takeaways

1. **Data Sources:**
   - Images table: Core content with metadata and tags
   - Users table: Profile information for analytics
   - Interactions table: Critical training signal (clicks, impressions, saves)

2. **Image Preprocessing:**
   - Resize to fixed dimensions (224x224) using letterbox strategy
   - Scale pixels from [0-255] to [0-1]
   - Apply z-score normalization with ImageNet statistics
   - Standardize color mode to RGB

3. **Feature Engineering:**
   - Work with learned embeddings (256-2048 dimensions)
   - Create training pairs/triplets from interaction data
   - Handle position bias in click data
   - Apply data augmentation for robustness

### Next Steps

In the next notebook, we'll cover:
- Model architecture selection (CNN, Vision Transformers)
- Training strategies (contrastive learning, triplet loss)
- Indexing and search infrastructure

In [ ]:
# Final Summary
print("\n" + "="*60)
print("DATA & FEATURE ENGINEERING SUMMARY")
print("="*60)

print("""
DATA SOURCES:
  - Images: 150B+ images with metadata & tags
  - Users: Profile data (demographics, location)
  - Interactions: Clicks, impressions, saves

PREPROCESSING PIPELINE:
  1. RGB conversion  -> Consistent color space
  2. Resize (224x224) -> Fixed input dimensions
  3. Scale (0-1)     -> Normalized range
  4. Z-score norm    -> Match pre-trained distribution

FEATURE ENGINEERING:
  - Embeddings: 256-2048 dim vectors
  - Training pairs from click interactions
  - Position bias correction
  - Data augmentation for robustness
""")